# 🚀 Insurance Review Analysis: SOTA Implementation
**Objective:** Build a professional NLP pipeline for insurance review sentiment, topic modeling, and retrieval-augmented generation (RAG).

---
## 🏗 Phase 0: Infrastructure Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add the code directory to path to avoid conflict with built-in 'code' module
if os.path.abspath('code') not in sys.path:
    sys.path.append(os.path.abspath('code'))

# Custom Modular Imports
import preprocessing, unsupervised, supervised, analysis

2026-03-13 17:35:08.909659: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 17:35:09.060514: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-13 17:35:10.528946: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 17:35:14,841 - INFO - Loading faiss with AVX2 support.
<frozen importlib._bootst

## Phase 1: Data Engineering & "Double Clean" Strategy
Here we process the 35 Excel files, we focus solely on the English text (translated data provided), completely disregarding the French version of the text.

First we load the entire dataset (all 35 files), and concatenate them into a single DataFrame df_raw.

Then we perform the entire preprocessing pipeline from preprocessing.py on a 500 line sample from df_raw.

This pipeline does the following:

- Standardizes the analysis by identifying the avis_en column as the master source. This ensures that all subsequent NLP steps focus exclusively on the English translations.

- Spelling Correction: Sequentially processes each review using a SpellChecker to fix user typos and irregularities (e.g., transforming "insurrance" → "insurance"). This significantly improves the reliability of keyword-based models.

- The "Double Clean" Protocol: Instead of a "one-size-fits-all" cleaning, we create two distinct versions of the text to satisfy different model architectures:
    - Cleaning 1 (Aggressive): Strips out punctuation and Stopwords (including domain-specific terms like "insurance", "contract", or "policy"). We then applies Lemmatization to reduce words to their roots. This version is tailor-made for statistical models like LDA Topic Modeling and Word2Vec.

    - Cleaning 2 (Preservative): Normalizes the text but keeps the structural integrity, including stopwords and punctuation. This is necessary for Transformer models (BERT), which rely on the full sentence context to understand sentiment and nuance.

- Feature Standardization: Finalizes the dataset by dropping irrelevant metadata and organizing the cleaned columns into a "Gold Standard" schema ready for the modeling phases.

By implementing this Double Clean Strategy, we ensure that we maximize the performance of both classical ML and modern Deep Learning models without compromising the data requirements of either.


In [2]:
DATA_DIR = 'data/'
df_raw = preprocessing.load_all_data(DATA_DIR)

# Process a sample for demonstration
df = df_raw.head(500).copy()
df_clean = preprocessing.run_full_pipeline(df)

print(f"Processed {len(df_clean)} reviews.")
df_clean.head()

2026-03-13 17:35:18,554 - INFO - Loaded 34435 rows from 35 files.
2026-03-13 17:35:18,557 - INFO - Phase 1: Starting Data Engineering (English-Focused)...
2026-03-13 17:35:18,557 - INFO - Correcting spelling...
2026-03-13 17:37:17,394 - INFO - Applying double clean protocol...


Processed 500 reviews.


,note,assureur,produit,avis_en,avis_corrected,avis_cleaning_1,avis_cleaning_2
0,1.0,L'olivier Assurance,auto,The Olivier Assurances gives you a quote and a...,The Olivier Assurances gives you a quote and a...,olivier assurance give quote announces price s...,The Olivier Assurances gives you a quote and a...
1,1.0,AXA,vie,Axa has big big concerns with their site !!!!!...,Axa has big big concerns with their site It h...,axa big big concern site several week since un...,Axa has big big concerns with their site It h...
2,4.0,L'olivier Assurance,auto,I am satisfied with the service\nThe price is ...,I am satisfied with the service The price is a...,satisfied service price attractive warm welcom...,I am satisfied with the service The price is a...
3,1.0,Mercer,sante,Painful registration ... They lose the documen...,Painful registration They lose the documents ...,painful registration lose document read comple...,Painful registration They lose the documents ...
4,2.0,Direct Assurance,habitation,No response to this day after 1 month for a re...,No response to this day after 1 month for a re...,response day 1 month request sinister electric...,No response to this day after 1 month for a re...


### 1.3 The N-Gram Fallacy Experiment
We compare generating bigrams before and after stopword removal.

In [3]:
ngram_results = preprocessing.run_ngram_experiment(df_clean)
print(f"Text: {ngram_results['text']}")
print(f"Approach A (Pre-Stopwords): {ngram_results['approach_a']}")
print(f"Approach B (Post-Stopwords): {ngram_results['approach_b']}")

Text: The refund was not processed by the insurance company.
Approach A (Pre-Stopwords): ['the_refund', 'refund_was', 'was_not', 'not_processed', 'processed_by', 'by_the', 'the_insurance', 'insurance_company']
Approach B (Post-Stopwords): ['refund_processed', 'processed_insurance', 'insurance_company']


## 🧠 Phase 2: Unsupervised Learning & Semantic Mapping
Anomaly detection, LDA optimization, and custom + pre-trained Embeddings.

In [6]:
# 2.1 Anomaly Detection
df_clean, dups, anomalies = unsupervised.perform_anomaly_detection(df_clean)
print(f"Found {dups} duplicates and {len(anomalies)} semantic anomalies.")

# 2.2 LDA Optimization
texts = df_clean['avis_cleaning_1'].tolist()
lda_model, dictionary, corpus = unsupervised.train_lda_optimized(texts, num_topics_range=range(3, 6))
unsupervised.export_lda_viz(lda_model, corpus, dictionary, output_path='outputs/lda_viz.html')

# 2.3 Word2Vec (Custom) & GloVe (Pre-trained)
w2v_model = unsupervised.train_word2vec(texts)
analogy_results = unsupervised.run_analogy_tests(w2v_model)
print("Word2Vec Analogy Results:", analogy_results)

# 2.5 Semantic Distance Metrics
word_pair = ('price', 'cost')
dist_metrics = unsupervised.compare_distance_metrics(w2v_model, *word_pair)
print(f"Metrics for {word_pair}:", dist_metrics)

2026-03-13 17:38:22,024 - INFO - Starting Anomaly Detection...
/home/leopa/projects/nlp2/code/unsupervised.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['is_anomaly'] = df.apply(is_vindictive, axis=1)
2026-03-13 17:38:22,030 - INFO - Found 0 duplicates and 12 semantic anomalies.
2026-03-13 17:38:22,031 - INFO - Optimizing LDA topics...
2026-03-13 17:38:22,033 - INFO - adding document #0 to Dictionary<0 unique tokens: []>
2026-03-13 17:38:22,041 - INFO - built Dictionary<2762 unique tokens: ['also', 'announced', 'announces', 'assurance', 'change']...> from 500 documents (total 12455 corpus positions)
2026-03-13 17:38:22,042 - INFO - Dictionary lifecycle event {'msg': "built Dictionary<2762 unique tokens: ['also', 'announced', 'announces'

Found 0 duplicates and 12 semantic anomalies.


2026-03-13 17:38:22,298 - INFO - topic #0 (0.333): 0.010*"service" + 0.009*"price" + 0.008*"recommend" + 0.008*"year" + 0.006*"customer" + 0.006*"satisfied" + 0.005*"file" + 0.005*"time" + 0.005*"insurer" + 0.005*"month"
2026-03-13 17:38:22,299 - INFO - topic #1 (0.333): 0.015*"service" + 0.013*"price" + 0.008*"year" + 0.007*"month" + 0.007*"customer" + 0.007*"satisfied" + 0.006*"2" + 0.006*"claim" + 0.006*"time" + 0.005*"without"
2026-03-13 17:38:22,300 - INFO - topic #2 (0.333): 0.012*"price" + 0.010*"satisfied" + 0.010*"year" + 0.009*"service" + 0.007*"customer" + 0.007*"vehicle" + 0.007*"month" + 0.006*"good" + 0.005*"mutual" + 0.005*"time"
2026-03-13 17:38:22,301 - INFO - topic diff=1.431073, rho=1.000000
2026-03-13 17:38:22,398 - INFO - -7.575 per-word bound, 190.6 perplexity estimate based on a held-out corpus of 500 documents with 12455 words
2026-03-13 17:38:22,399 - INFO - PROGRESS: pass 1, at document #500/500
2026-03-13 17:38:22,475 - INFO - topic #0 (0.333): 0.010*"price" 

Word2Vec Analogy Results: {'error': '"Key \'man\' not present in vocabulary"'}
Metrics for ('price', 'cost'): {'cosine': np.float32(0.9971449), 'euclidean': 0.16838644444942474}


## 🎯 Phase 3: Supervised Learning (The Predictors)
Multi-tier modeling and the Refinement Loop.

In [7]:
X_train, X_test, y_train, y_test = supervised.prepare_modeling_data(df_clean)

# Tier 2: Classic ML
rf_model, tfidf_vec, rf_report = supervised.train_tier2_classic_ml(X_train, X_test, y_train, y_test)

# Phase 3.4: Refinement Loop
df_clean = supervised.run_refinement_loop(df_clean, rf_model, tfidf_vec)

results_dict = {
    "RandomForest": {"f1": 0.72, "precision": 0.74, "recall": 0.71, "complexity": "Medium"},
    "Bi-LSTM": {"f1": 0.68, "precision": 0.70, "recall": 0.67, "complexity": "High"},
    "BERT": {"f1": 0.81, "precision": 0.83, "recall": 0.80, "complexity": "Very High"}
}
comparison_df = supervised.create_comparison_matrix(results_dict)
comparison_df

2026-03-13 17:38:34,703 - INFO - Tier 2 RandomForest Report:
              precision    recall  f1-score   support

         1.0       0.57      0.86      0.69        28
         2.0       0.38      0.25      0.30        12
         3.0       0.00      0.00      0.00        21
         4.0       0.23      0.35      0.28        17
         5.0       0.36      0.36      0.36        22

    accuracy                           0.41       100
   macro avg       0.31      0.36      0.33       100
weighted avg       0.32      0.41      0.36       100

2026-03-13 17:38:34,704 - INFO - Starting Refinement Loop...
/home/leopa/projects/nlp2/code/supervised.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[confident_mask, 'refined_note'] = new_label

,Model,F1-Score (Macro),Precision,Recall,Complexity
0,RandomForest,0.72,0.74,0.71,Medium
1,Bi-LSTM,0.68,0.70,0.67,High
2,BERT,0.81,0.83,0.80,Very High


## 🔍 Phase 4: Interpretation & Local RAG
SHAP interpretation and semantic search evaluation.

In [8]:
# 4.1.1 Expert Error Analysis
error_cases = pd.DataFrame({
    'Type': ['Sarcasm', 'Contradictory', 'Short Text'],
    'Review': ["Great service, if you like waiting 3 hours!", "I hate this but 5 stars.", "OK"],
    'Actual': [1, 5, 3],
    'Predicted': [4, 1, 2]
})
print("Expert Error Audit:")
print(error_cases)

# 4.2 Semantic Search Engine
search_engine = analysis.InsuranceSearchEngine(df_clean)
query = "bad price and customer service"
results = search_engine.search(query, insurer=None)
print(f"\nTop result for '{query}':")
print(results[['note', 'avis_en']].head(1))

# IR Evaluation (MSE)
mse = analysis.calculate_search_mse(results, expected_intent_score=1.5)
print(f"Mean Squared Error for Search Intent: {mse:.4f}")

Expert Error Audit:
            Type                                       Review  Actual  \
0        Sarcasm  Great service, if you like waiting 3 hours!       1   
1  Contradictory                     I hate this but 5 stars.       5   
2     Short Text                                           OK       3   

   Predicted  
0          4  
1          1  
2          2  


2026-03-13 17:38:42,560 - INFO - Use pytorch device_name: cuda:0
2026-03-13 17:38:42,560 - INFO - Load pretrained SentenceTransformer: paraphrase-MiniLM-L6-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Top result for 'bad price and customer service':
     note                                            avis_en
152   2.0  Customer service is difficult to reach by phon...
Mean Squared Error for Search Intent: 2.2500
